## Setup

In [1]:
import yaml
import pathlib
import pickle as pk

import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier

import utils
from losses import ReconstructionLoss
from data import MET_Data, get_transformation_function

In [2]:
def load_experiment(exp_type, base_dir, exp_name, checkpoints = False):
    exp_path = pathlib.Path(base_dir) / exp_name
    if exp_type == "cca":
        experiment = utils.load_pca_cca(exp_path)
    elif exp_type == "coupler":
        experiment = utils.load_coupler_folds(exp_path, get_checkpoints = checkpoints)
    elif exp_type == "autoencoder":
        experiment = utils.load_jit_folds(exp_path, get_checkpoints = checkpoints)
    else:
        raise ValueError(f'Experiment type "{exp_type}" not recognized.')
    return experiment

def get_ttype_mse(met_data, train_ids, test_ids):
    (mse, all_means) = ({}, {})
    merge_map = utils.get_tree_merge_map("../data/raw/mouse_VISp_gene_expression_matrices_2018-06-14/tree_Mouse_ALM-VISp_2018.csv", "n3")
    mappers = {merge: np.vectorize(lambda elem,merge=merge: merge_map[elem][merge]) 
               for merge in range(len(next(iter(merge_map.values()))) - 1)}
    for form in ["logcpm", "pca-ipfx", "arbors"]:
        train_data = met_data.query(train_ids, platforms = ["patchseq"], formats = [("logcpm", form)])
        test_data = met_data.query(test_ids, platforms = ["patchseq"], formats = [("logcpm", "pca-ipfx", "arbors")])
        (train_clusters, test_clusters) = (np.char.strip(train_data["cluster_label"]), np.char.strip(test_data["cluster_label"]))
        for (merge, map_func) in mappers.items():
            (train_clusters, test_clusters) = (map_func(train_clusters), map_func(test_clusters))
            (train_labels, test_labels) = (np.unique(train_clusters), np.unique(test_clusters))
            (train_masks, test_masks) = (train_labels[:, None] == train_clusters[None], test_labels[:, None] == test_clusters[None])
            label_means = {label: np.mean(train_data[form][mask], 0) for (label, mask) in zip(train_labels, train_masks)}
            label_mse = sum([np.square(label_means[label][None] - test_data[form][mask]).sum() 
                             for (label, mask) in zip(test_labels, test_masks)])
            mse.setdefault(form, []).append(label_mse / len(test_data["specimen_id"]))
            all_means[form] = {**label_means, **all_means.get(form, {})}
    return (mse, all_means)

def train_classifiers(encoder, classifiers, met_data, train_ids, test_ids, forms, trans_funcs, type_map, count_thresh):
    num_train = len(train_ids)
    joint_ids = np.concatenate([train_ids, test_ids])
    joint_data = met_data.get_specimens(joint_ids)
    joint_transformed = {form: trans_funcs.get(form, lambda x: x)(joint_data[form]) for form in forms}
    joint_tensors = {form: torch.from_numpy(array).float() for (form, array) in joint_transformed.items()}
    joint_z = encoder(joint_tensors)[0].numpy(force = True)
    
    joint_types = type_map(np.char.strip(joint_data["cluster_label"]))
    (train_types, test_types) = np.split(joint_types, [num_train])
    (labels, counts) = np.unique(train_types, return_counts = True)
    train_mask = np.isin(train_types, labels[counts > count_thresh])
    train_y = train_types[train_mask]
    test_mask = np.isin(test_types, train_y)
    test_y = test_types[test_mask]
    (train_z, test_z) = (joint_z[:num_train][train_mask], joint_z[num_train:][test_mask])
    
    scores = [model.fit(train_z, train_y).score(test_z, test_y) for model in classifiers]
    return scores

## VAE Models

In [3]:
met_data = MET_Data("../data/raw/MET_full_data.npz")

In [13]:
exp_info = {
    "var": {
        "type": "autoencoder",
        "dir": "../data/variational",
        "exps": ["t_arm", "e_arm", "m_arm"]},
    "tight": {
        "type": "autoencoder",
        "dir": "../data/variational_tight",
        "exps": ["t_e_arms", "t_m_arms", "e_m_arms", "met"]},
    "fixed": {
        "type": "autoencoder",
        "dir": "../data/rebalanced_fixed",
        "exps": ["met_2d_10", "met_2d_100", "met_2d_1000", "met_3d_10", "met_3d_100", "met_3d_1000", "met_10d_100"]},
    "full": {
        "type": "autoencoder",
        "dir": "../data/rebalanced_full_fixed",
        "exps": ["met_10d_100", "met_2d_100", "met_3d_100"]},
}

In [14]:
all_experiments = {f"{exp}-{group}": load_experiment(info["type"], info["dir"], exp) 
                   for (group, info) in exp_info.items() for exp in info["exps"]}

## Random Forest "Encoders"

### Raw

In [20]:
dest = pathlib.Path("../data/forest_baselines")
merge_map = utils.get_tree_merge_map("../data/raw/mouse_VISp_gene_expression_matrices_2018-06-14/tree_Mouse_ALM-VISp_2018.csv", "n3")
mappers = {merge: np.vectorize(lambda elem,merge=merge: merge_map[elem][merge]) 
           for merge in range(len(next(iter(merge_map.values()))) - 1)}
folds = next(iter(all_experiments.values()))["folds"]
for (fold, fold_dict) in folds.items():
    test_ids =  met_data.query(fold_dict["test_ids"], platforms = ["patchseq"], formats = [("logcpm", "pca-ipfx", "arbors")])["specimen_id"]
    for (modal, form) in [("T", "logcpm"), ("E", "pca-ipfx"), ("M", "arbors")]:
        train_ids = met_data.query(fold_dict["train_ids"], platforms = ["patchseq"], formats = [("logcpm", form)])["specimen_id"]
        accs = []
        for (map_id, map_func) in mappers.items():
            if map_id % 10 != 0:
                accs.append(np.nan)
            else:
                print(f"Generating Random Forest Fold {fold} - {map_id}: {form} -> T-type             ", end = "\r")
                classifiers = [RandomForestClassifier()]
                encoder = lambda form_dict: (torch.flatten(next(iter(form_dict.values())), start_dim = 1),)
                scores = train_classifiers(encoder, classifiers, met_data, train_ids, test_ids, [form], {}, map_func, 6)
                accs.append(scores[0])
                (dest / "models" / "raw_encoders" / str(fold) / str(map_id)).mkdir(parents = True, exist_ok = True)
                with open(dest / "models" / "raw_encoders" / str(fold) / str(map_id) / f"{modal}.pk", "wb") as target:
                    pk.dump(classifiers[0], target)
        (dest / "results" / "raw_encoders" / str(fold)).mkdir(parents = True, exist_ok = True)
        np.savez_compressed(dest / "results" / "raw_encoders" / str(fold) / f"{modal}.npz", np.asarray(accs))
    np.savez_compressed(dest / "models" / "raw_encoders" / str(fold) / "train_test_ids.npz", 
                        train = fold_dict["train_ids"], test = fold_dict["test_ids"])
print("\nComplete                                                       ")

Generating Random Forest Fold 10 - 100: arbors -> T-type               
Complete                                                       


### Latent

In [15]:
exps = ["t_arm-var", "e_arm-var", "m_arm-var"]

In [16]:
dest = pathlib.Path("../data/forest_baselines")
merge_map = utils.get_tree_merge_map("../data/raw/mouse_VISp_gene_expression_matrices_2018-06-14/tree_Mouse_ALM-VISp_2018.csv", "n3")
mappers = {merge: np.vectorize(lambda elem,merge=merge: merge_map[elem][merge]) 
           for merge in range(len(next(iter(merge_map.values()))) - 1)}
for exp_name in exps:
    exp_dict = all_experiments[exp_name]
    for (fold, fold_dict) in exp_dict["folds"].items():
        test_ids =  met_data.query(fold_dict["test_ids"], platforms = ["patchseq"], formats = [("logcpm", "pca-ipfx", "arbors")])["specimen_id"]
        for modal in exp_dict["config"]["modalities"]:
            forms = exp_dict["config"]["formats"][modal]
            train_ids = met_data.query(fold_dict["train_ids"], platforms = ["patchseq"], formats = [("logcpm", *forms)])["specimen_id"]
            accs = []
            for (map_id, map_func) in mappers.items():
                if map_id % 10 != 0:
                    accs.append(np.nan)
                    continue
                print(f"Generating {exp_name} Random Forest Fold {fold} - {map_id}: {forms} -> T-type             ", end = "\r")
                classifiers = [RandomForestClassifier()]
                encoder = fold_dict["best"][modal]["enc"]
                scores = train_classifiers(encoder, classifiers, met_data, train_ids, test_ids, forms, {}, map_func, 6)
                accs.append(scores[0])
                (dest / "models" / "latent_encoders" / exp_name / str(fold) / str(map_id)).mkdir(parents = True, exist_ok = True)
                with open(dest / "models" / "latent_encoders" / exp_name / str(fold) / str(map_id) / f"{modal}.pk", "wb") as target:
                    pk.dump(classifiers[0], target)
            (dest / "results" / "latent_encoders" / exp_name / str(fold)).mkdir(parents = True, exist_ok = True)
            np.savez_compressed(dest / "results" / "latent_encoders" / exp_name / str(fold) / f"{modal}.npz", np.asarray(accs))
        np.savez_compressed(dest / "models" / "latent_encoders" / exp_name / str(fold) / "train_test_ids.npz", 
                            train = fold_dict["train_ids"], test = fold_dict["test_ids"])
print("\nComplete                                                       ")

Generating m_arm-var Random Forest Fold 10 - 100: ['arbors'] -> T-type               
Complete                                                       


## T-type "Decoders"

In [13]:
dest = pathlib.Path("../data/forest_baselines")
folds = next(iter(all_experiments.values()))["folds"]
(ttype_mse, ttype_means) = ({}, {})
for (fold, fold_dict) in folds.items():
    print(f"Running fold {fold}     ", end = "\r")
    (train_ids, test_ids) = (fold_dict["train_ids"], fold_dict["test_ids"])
    try:
        (mse, means) = get_ttype_mse(met_data, train_ids, test_ids)
        for (form, merge_mses) in mse.items():
            (dest / "results" / "decoders" / str(fold)).mkdir(parents = True, exist_ok = True)
            np.savez_compressed(dest / "results" / "decoders" / str(fold) / f"{form}.npz", np.asarray(merge_mses))
        for (form, merge_means) in means.items():
            (dest / "models" / "decoders" / str(fold)).mkdir(parents = True, exist_ok = True)
            with open(dest / "models" / "decoders" / str(fold) / f"{form}.pk", "wb") as target:
                pk.dump(merge_means, target)
        np.savez_compressed(dest / "models" / "decoders" / str(fold) / "train_test_ids.npz", 
                            train = fold_dict["train_ids"], test = fold_dict["test_ids"])
    except KeyError as e:
        print(repr(e))

KeyError('Lamp5 Fam19a1 Tmem182')
KeyError('Lamp5 Krt73')
